'''
=================================================
Milestone 1

Nama  : Khoizuron Syakira
Batch : CODA-023-RMT

Deskripsi:
Project ini dibuat untuk menjalankan proses ETL (Extract, Transform, Load). 
1. Extract   : Melakukan web scraping data produk Fragrance (Parfum) dari 
               website Sephora Indonesia menggunakan Selenium dan BeautifulSoup.
2. Transform : Membersihkan data mentah hasil scraping menggunakan Pandas (menangani
               missing values dan mengubah tipe data numerik seperti harga).
3. Load      : Menyiapkan data bersih (CSV) yang siap dimasukkan ke dalam 
               database PostgreSQL.
=================================================
'''

In [6]:
import time
import pandas as pd
from selenium import webdriver
from bs4 import BeautifulSoup

# 1. Buka browser
driver = webdriver.Chrome()
url = 'https://www.sephora.co.id/categories/fragrance?view=60'
driver.get(url)
time.sleep(3) # Tunggu halaman awal terbuka

# 2. LOGIKA SCROLL OTOMATIS
# Menggulir layar ke bawah bertahap agar seluruh 60 produk dimuat oleh JavaScript
last_height = driver.execute_script("return document.body.scrollHeight")

for i in range(1, 10): # Scroll sebanyak 9-10 kali ke bawah
    driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight * {i/10});")
    time.sleep(1) # Jeda 1 detik per scroll agar browser sempat render data

# Scroll sekali lagi ke paling bawah untuk memastikan
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(2)

# 3. Ambil seluruh HTML setelah di-scroll penuh
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
driver.quit() 

# 4. Ambil kotak produk
# Pastikan menggunakan class kotak produk
product_cards = soup.find_all('div', {'class': 'product-card'}) # Atau cari class kotak produknya

brands, names, prices, ratings = [], [], [], []

for card in product_cards:
    try:
        brand_el = card.find('div', {'class': 'brand'})
        brands.append(brand_el.get_text(strip=True))
    except:
        brands.append(None)
        
    try:
        name_el = card.find('div', {'class': 'product-name'})
        names.append(name_el.get_text(strip=True))
    except:
        names.append(None)
        
    try:
        price_el = card.find('span', {'class': 'sell-price'})
        prices.append(price_el.get_text(strip=True))
    except:
        prices.append(None)
        
    try:
        rating_el = card.find('div', {'class': 'stars'})
        ratings.append(rating_el.get('style'))
    except:
        ratings.append(None)

# 5. Masukkan ke DataFrame
df_kotor = pd.DataFrame({
    'Brand': brands,
    'Nama_Produk': names,
    'Harga': prices,
    'Rating': ratings
})

print(f"Total data terambil: {len(df_kotor)} baris")
display(df_kotor.head())

# Simpan ke CSV
df_kotor.to_csv('sephora_fragrance_kotor.csv', index=False)

Total data terambil: 68 baris


,Brand,Nama_Produk,Harga,Rating
0,Armani Beauty,Sì Eau De Parfum Intense,Rp 3.141.000,--highlightedPercentage: 92%; --maxStars: '★★★...
1,Lancôme,Idôle Eau de Parfum,Rp 2.090.000,--highlightedPercentage: 96%; --maxStars: '★★★...
2,Yves Saint Laurent,Libre Eau De Parfum,Rp 3.500.000,--highlightedPercentage: 88.00000000000001%; -...
3,Yves Saint Laurent,Libre Le Parfum,Rp 3.820.000,--highlightedPercentage: 100%; --maxStars: '★★...
4,HERMÈS,Twilly d'Hermes Eau de Parfum,Rp 3.420.000,--highlightedPercentage: 90%; --maxStars: '★★★...


In [1]:
import pandas as pd

# 1. Baca data kotor hasil scraping
df = pd.read_csv('sephora_fragrance_kotor.csv')

# Cek kondisi awal data & missing values
print("--- Info Data Kotor ---")
df.info()

# 2. Hapus baris jika ada nama produk atau harga yang kosong
df = df.dropna(subset=['Nama_Produk', 'Harga'])

# 3. Ngebersihin Kolom 'Harga' (String -> Integer)
# Menghilangkan 'Rp', titik pemisah ribuan, dan spasi
df['Harga'] = df['Harga'].astype(str)
df['Harga'] = df['Harga'].str.replace('Rp', '', regex=False)
df['Harga'] = df['Harga'].str.replace('.', '', regex=False)
df['Harga'] = df['Harga'].str.strip()
df['Harga'] = pd.to_numeric(df['Harga'], errors='coerce').astype('Int64')

# 4. Ngebersihin Kolom 'Rating' (Mengambil angka persentase -> Skala 0 - 5)
# Mengambil angka sebelum tanda % pake Regex
df['Rating'] = df['Rating'].astype(str).str.extract(r'(\d+(?:\.\d+)?)%')[0]
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')

# Mengubah skala persentase (0-100%) jadi skala bintang (0.0 - 5.0)
df['Rating'] = (df['Rating'] / 20).round(1)

# 5. Cek hasil data yang udah bersih
print("\n--- Info Data Bersih ---")
df.info()
display(df.head(15))

# 6. Simpan ke file CSV bersih
# Sesuaikan penamaan file sesuai format tugas M1
df.to_csv('sephora_fragrance_bersih.csv', index=False)

--- Info Data Kotor ---
<class 'pandas.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Brand        68 non-null     str  
 1   Nama_Produk  68 non-null     str  
 2   Harga        68 non-null     str  
 3   Rating       67 non-null     str  
dtypes: str(4)
memory usage: 13.6 KB

--- Info Data Bersih ---
<class 'pandas.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Brand        68 non-null     str    
 1   Nama_Produk  68 non-null     str    
 2   Harga        68 non-null     Int64  
 3   Rating       67 non-null     float64
dtypes: Int64(1), float64(1), str(2)
memory usage: 5.1 KB


,Brand,Nama_Produk,Harga,Rating
0,Armani Beauty,Sì Eau De Parfum Intense,3141000,4.6
1,Lancôme,Idôle Eau de Parfum,2090000,4.8
2,Yves Saint Laurent,Libre Eau De Parfum,3500000,4.4
3,Yves Saint Laurent,Libre Le Parfum,3820000,5.0
4,HERMÈS,Twilly d'Hermes Eau de Parfum,3420000,4.5
5,DIOR,New Miss Dior Blooming Bouquet Eau De Toilette,1480000,4.8
6,DIOR,Miss DIOR Essence De Parfum,3090000,5.0
7,Yves Saint Laurent,Black Opium Eau de Parfum,3500000,4.6
8,Jo Malone London,Peony & Blush Suede Cologne,520000,4.6
9,Yves Saint Laurent,Libre Berry Crush Eau De Parfum,3500000,4.7
